# 02 · Features

Construye las tablas que usarán el baseline (03) y el modelo (04).

**Parte 1 (este notebook, primera sección): puntos de K y D/ST.** nflverse trae `fantasy_points_ppr` para QB/RB/WR/TE, pero no para kickers ni defensas. Los calculo con `config/scoring.yaml` y **valido el cálculo contra los puntos reales de ESPN** en 2026.

**Parte 2 (siguiente sección): features** de forma reciente, uso, contexto del partido y rival, sin fuga de información hacia el futuro.

> Solo temporada regular. Las credenciales se leen de `.env` y nunca se imprimen.

## 1. Setup

In [ ]:
import os
from pathlib import Path

import polars as pl
import yaml
import nflreadpy as nfl
from dotenv import load_dotenv
from espn_api.football import League
from espn_api.football.constant import PRO_TEAM_MAP

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
CONFIG = ROOT / "config"
DATA_PROC.mkdir(parents=True, exist_ok=True)
load_dotenv(ROOT / ".env")

SCORING = yaml.safe_load((CONFIG / "scoring.yaml").read_text(encoding="utf-8"))
VALIDATION = yaml.safe_load((CONFIG / "validation.yaml").read_text(encoding="utf-8"))

SEASONS = [2023, 2024, 2025, 2026]
REFRESH = False  # True para volver a descargar (2026 está en curso: refrescar cada semana)

pl.Config.set_tbl_rows(20)

### Esquema de validación (`config/validation.yaml`)

Walk-forward por temporada: cada fold entrena **solo** con temporadas anteriores a la que valida. La liga es nueva en 2026, así que solo 2026 tiene proyecciones de ESPN: en 2023–2025 el modelo se compara con un baseline simple (media móvil), y contra ESPN solo en 2026.

In [ ]:
pl.DataFrame([
    {"fold": f["name"], "entrena": ", ".join(map(str, f["train"])), "valida": str(f["valid"]), "compara contra": VALIDATION["baseline"]}
    for f in VALIDATION["folds"]
] + [{"fold": "holdout", "entrena": ", ".join(map(str, VALIDATION["holdout"]["train"])),
      "valida": str(VALIDATION["holdout"]["season"]), "compara contra": VALIDATION["holdout"]["compare_against"]}])

## 2. Datos

In [ ]:
def cached(name, loader):
    """Lee data/raw/<name>.parquet o lo descarga con `loader` si no existe (o si REFRESH)."""
    path = DATA_RAW / f"{name}.parquet"
    if path.exists() and not REFRESH:
        return pl.read_parquet(path)
    df = loader()
    df.write_parquet(path)
    return df

player_stats = cached("player_stats_weekly_2023_2026", lambda: nfl.load_player_stats(SEASONS, summary_level="week"))
team_stats = cached("team_stats_weekly_2023_2026", lambda: nfl.load_team_stats(SEASONS, summary_level="week"))
schedules = cached("schedules_2023_2026", lambda: nfl.load_schedules(SEASONS))

for name, df in [("player_stats", player_stats), ("team_stats", team_stats), ("schedules", schedules)]:
    print(f"{name:13s} {df.height:>7,} filas")

## 3. Puntos de kicker

| regla | columnas de nflverse |
|---|---|
| FG 0–39 (3) | `fg_made_0_19` + `fg_made_20_29` + `fg_made_30_39` |
| FG 40–49 (4), 50–59 (5), 60+ (6) | `fg_made_40_49`, `fg_made_50_59`, `fg_made_60_` |
| FG fallado (−1) | `fg_missed` **+ `fg_blocked`** |
| PAT (1) | `pat_made` |

ESPN cuenta un FG bloqueado como fallado; en nflverse `fg_missed` no incluye los bloqueados. Lo descubrí en la validación contra ESPN (sección 5).

In [ ]:
KICK = {r["abbr"]: r["points"] for r in SCORING["kicking"]}
FG_MADE = {"FG0": ["fg_made_0_19", "fg_made_20_29", "fg_made_30_39"],
           "FG40": ["fg_made_40_49"], "FG50": ["fg_made_50_59"], "FG60": ["fg_made_60_"]}
K_COLS = [c for cols in FG_MADE.values() for c in cols] + ["fg_att", "fg_made", "fg_missed", "fg_blocked", "pat_att", "pat_made"]

points_k = (player_stats
    .filter(pl.col("season_type") == "REG", pl.col("position") == "K")
    .with_columns(pl.col(K_COLS).fill_null(0))
    .with_columns(fantasy_points=(
        sum(pl.sum_horizontal(cols) * KICK[abbr] for abbr, cols in FG_MADE.items())
        + (pl.col("fg_missed") + pl.col("fg_blocked")) * KICK["FGM"]
        + pl.col("pat_made") * KICK["PAT"]))
    .select("season", "week", "game_id", "player_id", "player_display_name", "team", "opponent_team",
            "fg_att", "fg_made", "fg_missed", "fg_blocked", "pat_att", "pat_made", pl.col("fantasy_points").cast(pl.Float64)))

print(f"{points_k.height:,} partidos de kicker")
points_k.group_by("season").agg(pl.len().alias("partidos"), pl.col("fantasy_points").mean().round(2).alias("media"),
                                 pl.col("fantasy_points").max().alias("max")).sort("season")

## 4. Puntos de D/ST

**Eventos** (de `team_stats` del propio equipo): sacks, INTs, fumbles recuperados, safeties, bloqueos y TDs. En los TDs, nflverse separa `def_tds`, `fumble_recovery_tds` y `special_teams_tds`, así que sumo las tres columnas.

**Puntos y yardas permitidas:** las calculo con la definición de ESPN, que obtuve comparando con los valores reales de ESPN en la sección 5:
- **Puntos permitidos** = marcador del rival − 6 × (TDs de su defensa + TDs tras recuperar un fumble) − 2 × sus safeties. ESPN no le cuenta a tu D/ST los puntos que anota la defensa rival contra tu ofensiva.
- **Yardas permitidas** = yardas de pase del rival − yardas perdidas en sacks + yardas de carrera. Ojo: en nflverse `sack_yards_lost` viene con signo negativo, por eso uso el valor absoluto.

In [ ]:
DST = {r["abbr"]: r["points"] for r in SCORING["dst"]["events"]}

def bucket_points(col, rules):
    """Puntos según rangos inclusivos {min, max}; max=None = sin límite superior."""
    expr = pl.lit(None, dtype=pl.Float64)
    for r in reversed(rules):
        in_range = pl.col(col) >= r["min"] if r["max"] is None else pl.col(col).is_between(r["min"], r["max"])
        expr = pl.when(in_range).then(pl.lit(float(r["points"]))).otherwise(expr)
    return expr

TS_COLS = ["def_sacks", "def_interceptions", "fumble_recovery_opp", "def_safeties", "def_punt_blocks", "def_fg_blocks",
           "def_pat_blocks", "def_tds", "fumble_recovery_tds", "special_teams_tds", "def_2pt_made",
           "passing_yards", "sack_yards_lost", "rushing_yards"]
ts = (team_stats.filter(pl.col("season_type") == "REG")
      .select("season", "week", "team", *TS_COLS).with_columns(pl.col(TS_COLS).fill_null(0)))

# Un registro por equipo y partido jugado (sin byes ni partidos sin marcador)
games = schedules.filter(pl.col("game_type") == "REG", pl.col("home_score").is_not_null())
sides = pl.concat([
    games.select("season", "week", "game_id", team="home_team", opponent="away_team", opp_score="away_score"),
    games.select("season", "week", "game_id", team="away_team", opponent="home_team", opp_score="home_score"),
])
opp = ts.select("season", "week", opponent="team",
                opp_pass_yds="passing_yards", opp_sack_yds="sack_yards_lost", opp_rush_yds="rushing_yards",
                opp_def_tds="def_tds", opp_fr_tds="fumble_recovery_tds", opp_safeties="def_safeties")

points_dst = (sides
    .join(ts.drop("passing_yards", "sack_yards_lost", "rushing_yards"), on=["season", "week", "team"], how="left")
    .join(opp, on=["season", "week", "opponent"], how="left")
    .with_columns(
        points_allowed=pl.col("opp_score") - 6 * (pl.col("opp_def_tds") + pl.col("opp_fr_tds")) - 2 * pl.col("opp_safeties"),
        yards_allowed=pl.col("opp_pass_yds") - pl.col("opp_sack_yds").abs() + pl.col("opp_rush_yds"),
        tds=pl.col("def_tds") + pl.col("fumble_recovery_tds") + pl.col("special_teams_tds"),
        blocks=pl.col("def_punt_blocks") + pl.col("def_fg_blocks") + pl.col("def_pat_blocks"))
    .with_columns(
        pa_points=bucket_points("points_allowed", SCORING["dst"]["points_allowed"]),
        ya_points=bucket_points("yards_allowed", SCORING["dst"]["yards_allowed"]),
        event_points=(pl.col("def_sacks") * DST["SK"] + pl.col("def_interceptions") * DST["INT"]
                      + pl.col("fumble_recovery_opp") * DST["FR"] + pl.col("def_safeties") * DST["SF"]
                      + pl.col("blocks") * DST["BLKK"] + pl.col("tds") * DST["INTTD"]  # todos los TDs valen 6
                      + pl.col("def_2pt_made") * DST["2PTRET"]))
    .with_columns(fantasy_points=(pl.col("pa_points") + pl.col("ya_points") + pl.col("event_points")).cast(pl.Float64))
    .select("season", "week", "game_id", "team", "opponent", "points_allowed", "yards_allowed",
            "def_sacks", "def_interceptions", "fumble_recovery_opp", "def_safeties", "blocks", "tds",
            "pa_points", "ya_points", "event_points", "fantasy_points")
    .sort("season", "week", "team"))

n_null = points_dst["fantasy_points"].null_count()
print(f"{points_dst.height:,} partidos de D/ST · sin datos de team_stats: {n_null}")
assert n_null == 0, "Hay partidos sin team_stats: revisar joins"
points_dst.group_by("season").agg(pl.len().alias("partidos"), pl.col("fantasy_points").mean().round(2).alias("media"),
                                   pl.col("fantasy_points").min().alias("min"), pl.col("fantasy_points").max().alias("max")).sort("season")

## 5. Validación contra los puntos reales de ESPN (2026)

Para cada K y cada D/ST de la NFL descargo con `league.player_info` los puntos que ESPN les asignó en cada semana ya jugada de 2026 y los comparo con mi cálculo. Guardo también el detalle de ESPN (puntos y yardas permitidas, sacks) para poder explicar cada diferencia.

Así descubrí las tres reglas de las secciones 3 y 4 (FG bloqueado = fallado; ESPN excluye los TDs y safeties de la defensa rival de los puntos permitidos; signo de `sack_yards_lost`).

In [ ]:
league = League(league_id=int(os.environ["ESPN_LEAGUE_ID"]), year=2026,
                espn_s2=os.environ["ESPN_S2"], swid=os.environ["ESPN_SWID"])

# IDs de ESPN: los D/ST usan -16000 - proTeamId; los kickers se mapean con ff_playerids
ESPN_TO_NFLVERSE = {"WSH": "WAS", "LAR": "LA"}
dst_ids = pl.DataFrame([{"espn_id": -16000 - tid, "team": ESPN_TO_NFLVERSE.get(abbr, abbr)}
                        for tid, abbr in PRO_TEAM_MAP.items() if tid != 0])
ff_ids = nfl.load_ff_playerids().select("espn_id", "gsis_id").drop_nulls().unique("gsis_id")

k26 = points_k.filter(pl.col("season") == 2026).join(ff_ids, left_on="player_id", right_on="gsis_id", how="left")
WEEKS = sorted(points_dst.filter(pl.col("season") == 2026)["week"].unique().to_list())
print(f"Semanas 2026 jugadas: {WEEKS} · K sin espn_id: {k26['espn_id'].null_count()}")

In [ ]:
actuals_path = DATA_RAW / "espn_actuals_k_dst_2026.parquet"
need_fetch = REFRESH or not actuals_path.exists() or \
             sorted(pl.read_parquet(actuals_path)["week"].unique().to_list()) != WEEKS

if need_fetch:
    ids = k26["espn_id"].drop_nulls().unique().to_list() + dst_ids["espn_id"].to_list()
    rows = []
    for i in range(0, len(ids), 25):
        res = league.player_info(playerId=ids[i:i + 25])
        for p in res if isinstance(res, list) else [res]:
            for wk in WEEKS:
                wk_stats = p.stats.get(wk, {})
                if "points" not in wk_stats:
                    continue
                raw = wk_stats.get("breakdown", {})
                rows.append({"espn_id": p.playerId, "name": p.name, "pos": p.position, "week": wk,
                             "espn_points": float(wk_stats["points"]),
                             "espn_pa": raw.get("defensivePointsAllowed"), "espn_ya": raw.get("defensiveYardsAllowed"),
                             "espn_sacks": raw.get("defensiveSacks")})
    actuals = pl.DataFrame(rows, schema_overrides={"espn_pa": pl.Float64, "espn_ya": pl.Float64, "espn_sacks": pl.Float64})
    actuals.write_parquet(actuals_path)
else:
    actuals = pl.read_parquet(actuals_path)
actuals.group_by("pos").agg(pl.len().alias("partidos"), pl.col("espn_id").n_unique().alias("jugadores"))

In [ ]:
TOL = 1e-6
MIN_MATCH = 0.95  # por debajo de esto la fórmula está mal, no son simples correcciones de datos

val_k = (actuals.filter(pl.col("pos") == "K")
         .join(k26.select("espn_id", "week", "player_display_name", "fg_missed", "fg_blocked", "fantasy_points"),
               on=["espn_id", "week"], how="left")
         .with_columns(diff=pl.col("espn_points") - pl.col("fantasy_points")))
val_dst = (actuals.filter(pl.col("pos") == "D/ST").join(dst_ids, on="espn_id")
           .join(points_dst.filter(pl.col("season") == 2026), on=["team", "week"], how="left")
           .with_columns(diff=pl.col("espn_points") - pl.col("fantasy_points")))

summary = []
for name, v in [("K", val_k), ("D/ST", val_dst)]:
    ok = (v["diff"].abs() < TOL).sum()
    summary.append({"posición": name, "partidos": v.height, "coinciden": ok, "pct": round(ok / v.height * 100, 1),
                    "sin cálculo": v["fantasy_points"].null_count()})
summary.append({"posición": "D/ST: puntos permitidos", "partidos": val_dst.height,
                "coinciden": (val_dst["points_allowed"] == val_dst["espn_pa"]).sum(), "pct": None, "sin cálculo": None})
summary.append({"posición": "D/ST: yardas permitidas", "partidos": val_dst.height,
                "coinciden": (val_dst["yards_allowed"] == val_dst["espn_ya"]).sum(), "pct": None, "sin cálculo": None})
summary = pl.DataFrame(summary)

for row in summary.filter(pl.col("pct").is_not_null()).iter_rows(named=True):
    if row["pct"] / 100 < MIN_MATCH:
        raise ValueError(f"{row['posición']}: solo {row['pct']}% coincide con ESPN: revisar la fórmula")
summary

Diferencias restantes y su causa probable:

In [ ]:
(pl.concat([
    val_k.filter(pl.col("diff").abs() > TOL)
         .select(pl.col("name"), "week", "espn_points", "fantasy_points", "diff",
                 detalle=pl.format("fg_missed={} fg_blocked={}", "fg_missed", "fg_blocked")),
    val_dst.filter(pl.col("diff").abs() > TOL)
           .select(pl.col("name"), "week", "espn_points", "fantasy_points", "diff",
                   detalle=pl.format("sacks nflverse={} / ESPN={} · PA {} / {} · YA {} / {}",
                                     "def_sacks", "espn_sacks", "points_allowed", "espn_pa", "yards_allowed", "espn_ya")),
], how="vertical_relaxed"))

Si solo queda alguna diferencia puntual en una estadística (por ejemplo, un sack de más en ESPN) y los puntos y yardas permitidas cuadran, es una discrepancia entre las fuentes de datos o una corrección de estadísticas posterior, no un error de la fórmula.

## 6. Guardar

In [ ]:
points_dst = points_dst.join(dst_ids, on="team", how="left")  # espn_id para cruzar con rosters de ESPN
points_k = points_k.join(ff_ids.rename({"gsis_id": "player_id"}), on="player_id", how="left")

points_k.write_parquet(DATA_PROC / "points_k.parquet")
points_dst.write_parquet(DATA_PROC / "points_dst.parquet")
for p in ["points_k.parquet", "points_dst.parquet"]:
    print(f"✓ data/processed/{p}")

## Supuestos y limitaciones

- **Las líneas de apuestas son de cierre.** `schedules` de nflverse trae el spread y el total *de cierre* (justo antes del partido). Si la alineación se decide días antes, solo estarán las líneas de apertura o las del momento, así que el backtest es algo optimista respecto al uso real.
- **El modelo supone que el jugador juega.** El objetivo solo existe para partidos jugados: el modelo predice los puntos *si juega*, no la probabilidad de que juegue. Lesiones, inactivos y descansos se manejan fuera del modelo (con el estado de lesión de ESPN al decidir la alineación).
- **Validación de K y D/ST limitada a 2026.** La fórmula se verificó contra ESPN solo en las semanas jugadas de 2026, porque la liga es nueva. Para 2023–2025 se asume que las reglas de ESPN no cambiaron.
- **Casos raros de D/ST sin verificar:** si ESPN resta de los puntos permitidos los TDs de retorno (kickoff o punt) del rival, y cómo registra nflverse los retornos de 2 pts (`def_2pt_made`). No aparecieron en la muestra de validación. La safety de 1 punto no se modela.
- **Correcciones de estadísticas.** ESPN y nflverse pueden diferir en algún dato puntual (por ejemplo, sacks compartidos o corregidos después del partido).